In [ ]:
import random
from typing import List, Dict
from langchain_community.chat_models import ChatOllama
from langchain.agents import initialize_agent, AgentType
from persona import generate_persona, Persona  # 參考您提供的 persona.py

# === 1. 實驗參數設定 (依據論文實驗設計) ===
NUM_AGENTS = 5  # 模擬的代理人數量
SIMULATION_ROUNDS = 5  # 模擬對話的輪數
REPETITIONS = 25       # 重複 25 次以確保數據穩定
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議" # 論文常見的社會議題範例

# 實驗1 固定model
FIRST_TEST_MODEL = "llama3:8B"

# === 2. 初始化 LLM 與角色 (Persona) ===
# 論文中使用不同的 LLM 來模擬多樣性
llm_pool = [
    ChatOllama(model="llama3:8B"),
    ChatOllama(model="mistral:7B"),
    ChatOllama(model="gemma3:4b")
]

def initialize_social_simulation():
    agents = {}
    personas = {}
    
    # 生成具備初始立場的角色
    # 立場分配：確保 2 支持 / 2 反對 / 1 中立，避免隨機偏誤
    controlled_stances = ["支持", "支持", "反對", "反對", "中立"]
    random.shuffle(controlled_stances)
    
    for i in range(NUM_AGENTS):
        name = f"Agent_{i+1}"
        initial_stance = controlled_stances[i]
        
        # 使用 persona.py 的生成功能
        p = generate_persona(name=name, initial_stance=initial_stance)
        personas[name] = p
        
        # 綁定 LLM
        selected_llm = ChatOllama(model=FIRST_TEST_MODEL)
        
        # 建立 Agent
        agents[name] = initialize_agent(
            tools=[], # 論文實驗通常聚焦於對話，不一定使用外部工具
            llm=selected_llm,
            handle_parsing_errors=True,
            agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
            verbose=False,
            agent_kwargs={
                "prefix": p.to_prompt(), # 注入論文核心的 Persona 描述
            }
        )
    return agents, personas

# === 3. 社會模擬流程 (模擬論文中的對話機制) ===
def run_social_influence_simulation():
    agents, personas = initialize_social_simulation()
    conversation_history = []
    
    print(f"=== 模擬開始：關於 {TOPIC} 的討論 ===")
    
    for r in range(SIMULATION_ROUNDS):
        print(f"\n--- 第 {r+1} 輪對話 ---")
        
        # 論文中通常採用隨機輪詢或順序輪詢
        agent_names = list(agents.keys())
        random.shuffle(agent_names)
        
        for name in agent_names:
            # 建立 Context：包含之前的對話摘要（論文中的 Short-term Memory）
            # context = "\n".join(conversation_history[-5:]) # 取最近5條記錄
            full_context = "\n".join(conversation_history)
            
            prompt = f"""
            目前的討論話題是：{TOPIC}。
            這是目前的討論進度：
            {full_context}
            
            請根據你的角色人格(Persona)與立場，對目前的討論做出回應。
            你不需要表現得像 AI，請像一個真實的人類在社交平台發言。
            """
            
            response = agents[name].run(input=prompt, chat_history=[])
            
            formatted_entry = f"{name} ({personas[name].initial_stance}): {response}"
            conversation_history.append(formatted_entry)
            print(formatted_entry)

    # === 4. 立場變化分析 (論文核心分析點) ===
    analyze_opinion_shift(agents, personas, conversation_history)

def analyze_opinion_shift(agents, personas, history):
    print("\n=== 實驗結果：意見偏好分析 ===")
    # 論文會透過 LLM 作為評審 (LLM-as-a-judge) 來判斷代理人對話後的立場是否改變
    analyzer = ChatOllama(model="llama3:8B")
    
    for name, persona in personas.items():
        analysis_prompt = f"""
        角色：{persona.name}
        初始立場：{persona.initial_stance}
        
        以下是他在討論中的發言：
        {[h for h in history if h.startswith(name)]}
        
        請判斷他在討論結束後，立場是否有發生「從眾效應」或「意見極化」？
        請簡短回答。
        """
        result = analyzer.invoke(analysis_prompt)
        print(f"[{name}] 立場分析：{result.content}")

if __name__ == "__main__":
    run_social_influence_simulation()

=== 模擬開始：關於 關於數位身分證(eID)推行的隱私與便利性爭議 的討論 ===

--- 第 1 輪對話 ---


C:\Users\TUF\AppData\Local\Temp\ipykernel_25800\4007061647.py:82: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agents[name].run(input=prompt, chat_history=[])


Agent_2 (支持): 你說得對！數位身分證的討論實在太重要了，尤其是在便利性與隱私這兩方面。我同意，政府需要更透明地解釋數據管理制度，而且必須建立嚴密的監督機制，防止濫用。 

    我個人認為，這確實需要多方討論，提出更具體的建議，例如資料加密技術的應用、個人資料的賦予權利、以及定期檢視制度的有效性等等。  重要的是，要讓民眾對數位身分證的應用有充分的了解，並能夠參與相關的討論，共同決定這個政策的走向。
Agent_5 (反對): I totally agree with you. The discussion on eID is indeed crucial, especially considering the aspects of convenience and privacy. I believe that the government needs to be more transparent in explaining their data management system and should establish a strict monitoring mechanism to prevent misuse. 

Personally, I think we need to have a multi-party discussion and propose concrete suggestions such as the application of data encryption technology, individual data ownership rights, and periodic review of the system's effectiveness. It is essential that the public understands the use of eID and has the opportunity to participate in related discussions to jointly determine the direction of this policy.
Agent_3 (支持): 這場討論實在太有意義了！Agent_2 和 Agent_5 都提出了非常重要的點，數位身分證的便利性和隱私